# Sentiment-to-Price Correlation Analyzer - EDA & Prototyping

This notebook demonstrates the end-to-end workflow for analyzing the relationship between National Stock Exchange of India (NSE) stock news sentiment and next-day price movements.

## Workflow Overview:
1. **Data Acquisition**: Fetch 20 NSE stock price histories (`yfinance`) and headline news (`growfin` / RSS / yfinance).
2. **Sentiment Analysis**: FinBERT inference (`ProsusAI/finbert`) for positive, neutral, negative probability scores.
3. **Feature Engineering**: Construct lag features, rolling metrics, and binary return targets.
4. **Correlation Analysis**: Pearson $r$ and $p$-value statistical evaluation.
5. **Predictive Modeling**: XGBoost binary classifier evaluated against Random and Same-as-Yesterday baselines.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is on sys.path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data_collector import DataCollector
from src.sentiment_analyzer import SentimentAnalyzer
from src.feature_engineering import FeatureEngineer
from src.correlation_analyzer import CorrelationAnalyzer
from src.prediction_model import SentimentPredictor

print("All modules loaded successfully!")

## 1. Fetch Raw Market Prices and Financial News

In [ ]:
collector = DataCollector(days_history=60)
raw_prices = collector.fetch_stock_prices()
raw_news = collector.fetch_stock_news()

print(f"Raw Prices Shape: {raw_prices.shape}")
print(f"Raw News Shape: {raw_news.shape}")
display(raw_prices.head())
display(raw_news.head())

## 2. FinBERT Sentiment Scoring

In [ ]:
analyzer = SentimentAnalyzer()
news_sentiment = analyzer.analyze_dataframe(raw_news)
display(news_sentiment[['Date', 'Symbol', 'Headline', 'sentiment_label', 'confidence', 'sentiment_score']].head(10))

## 3. Daily Aggregation & Feature Matrix Construction

In [ ]:
fe = FeatureEngineer(raw_prices, news_sentiment)
processed_data = fe.merge_and_build_features()
print(f"Processed Feature Dataset Shape: {processed_data.shape}")
display(processed_data[['Date', 'Symbol', 'Close', 'avg_sentiment', 'sentiment_lag1', 'sentiment_ma5', 'next_day_return', 'target_up']].head())

## 4. Statistical Correlation Evaluation

In [ ]:
ca = CorrelationAnalyzer(processed_data)
overall_stats, stock_corrs = ca.compute_correlations()
print(f"Overall Correlation r: {overall_stats['r']}, p-value: {overall_stats['p_value']}")
display(stock_corrs.head(10))

## 5. Visualizing Correlations & Time Series Overlays

In [ ]:
ca.plot_correlation_bar_chart(stock_corrs)
ca.plot_scatter_with_regression()
ca.plot_time_series_overlay(sample_stock="RELIANCE")

## 6. XGBoost Model Training vs Baselines

In [ ]:
sp = SentimentPredictor(processed_data)
results_df, xgb_predictions = sp.evaluate()
display(results_df)